# GenAI-Kickstart
## Lesson: Agents

This lesson is intended to show you how to build an agentic system using Google ADK (Agent Development Kit).  
In the previous lesson (Tools) you learned how to define and execute tool calls manually. Here, an **agent** handles the tool-calling loop for you automatically.

During this lesson you will learn how to ...
- define a simple agent with tools
- use the RAG system as a tool for a new agent (agentic RAG)
- integrate external tools via MCP (Model Context Protocol)

### Set up the environment
Import the necessary libraries, set constants, and define helper functions.

In [6]:
import sys
sys.path.append('../../../')

import os
import uuid
import warnings
import logging

from chromadb import EphemeralClient
from dotenv import load_dotenv, find_dotenv
from google.genai import types
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

from lib.validation import validate_environment_for_model
from lib.constants import PROCESSED_DATA_PATH
from lib.rag import do_embed, do_ingestion
from lib.evaluation import FetchedChunk, do_top_k_fetching

# Ignore all warnings and update logging
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

load_dotenv(find_dotenv())

True

In [7]:
# Models
GENERATION_PROVIDER = os.getenv("DEFAULT_GENERATION_PROVIDER")
GENERATION_MODEL = os.getenv("DEFAULT_GENERATION_MODEL")
EMBEDDING_PROVIDER = os.getenv("DEFAULT_EMBEDDING_PROVIDER")
EMBEDDING_MODEL = os.getenv("DEFAULT_EMBEDDING_MODEL")

# Set the full model names
generation_model = f"{GENERATION_PROVIDER}/{GENERATION_MODEL}"
embedding_model = f"{EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}"

# ADK model — uses LiteLlm wrapper so any LiteLLM-supported model works
adk_model = LiteLlm(model=generation_model)

# Set defaults for retrieval
DEFAULT_TOP_K = 3
DEFAULT_CHUNK_OVERLAP = 100
DEFAULT_CHUNK_SIZE = 2000

# Paths
BOOK_DB = PROCESSED_DATA_PATH / "books.db"

# Validate environment
validate_environment_for_model(model=generation_model)
validate_environment_for_model(model=embedding_model)

In [8]:
# Initialize the chromadb client
chromadb_client = EphemeralClient()

In [9]:
# This might be useful for a tool we need to create later
def do_retrieval(user_input: str, top_k: int = DEFAULT_TOP_K) -> list[FetchedChunk]:
    """Retrieves the k semantically closest chunks to the user input from the knowledgebase."""
    user_input_embedding = do_embed(user_input, embedding_model)
    return do_top_k_fetching(user_input_embedding, top_k, chromadb_client)


KNOWLEDGEBASE_CONTENT = [
    "study_in_scarlett.txt",
    "adventures_of_sherlock_holmes.txt",
]

do_ingestion(
    file_names=KNOWLEDGEBASE_CONTENT,
    data_path=PROCESSED_DATA_PATH,
    embedding_model=embedding_model,
    chromadb_client=chromadb_client,
)

#### Helpers for agents

In [10]:
# We need this for the conversation
USER_ID = "user_1"

# We will need a session to store the conversation
session_service = InMemorySessionService()

def create_random_session_id() -> str:
    """Creates a random session ID."""
    return str(uuid.uuid4())

async def call_agent_async(query: str, runner, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n<<< User Query: {query}")

  # Prepare the user's message in ADK format
  content = types.Content(role='user', parts=[types.Part(text=query)])
  final_response_text = "Agent did not produce a final response." # Default

  async for event in runner.run_async(user_id=USER_ID, session_id=session_id, new_message=content):
      # Log tool calls and results from the event stream
      if event.content and event.content.parts:
          for part in event.content.parts:
              if part.function_call:
                  print(f"\n>>> Tool Call: {part.function_call.name}({part.function_call.args})")
              if part.function_response:
                  response = part.function_response.response
                  # Extract the actual result from different response formats
                  if isinstance(response, dict):
                      result_str = str(response.get("result", response))
                  else:
                      result_str = str(response)
                  if len(result_str) > 200:
                      result_str = result_str[:200] + "..."
                  print(f"<<< Tool Result: {result_str}")

      if event.is_final_response():
          if event.content and event.content.parts:
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate:
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          break

  print(f"\n<<< Agent Response: {final_response_text}")

async def run_conversation(session_id: str, user_prompt: str, runner: Runner):

    await call_agent_async(
        query=user_prompt,
        runner=runner,
        session_id=session_id
        )

# Exercise 1: Your first agent

In this exercise you will create your first agent. The agent is a virtual employee of a bookstore. The agent should be able to use the function below as a tool to look up information about books in the catalog.  
Your task is to  
- create a docstring for the `read_book_database` function for giving the agent more information about what this function does
- provide the tool to the agent
- provide a general description and instruction (aka. system prompt) to the agent.


In [11]:
# TODO: Provide a meaningful description within the docstring. You can execute the function to see the output and use this for the description.
def read_book_database() -> str:
  """Tool for getting information about all books of the bookstore including the book titles, authors and ISBN numbers.

  Returns:
      str: The content of the book database, including ISBN numbers.
  """
  with open(BOOK_DB, "r", encoding="utf-8") as f:
    return f.read()

In [12]:
# TODO: Provide the tools, description and instruction (system prompt) to the bookstore agent
simple_bookstore_agent = Agent(
    name="simple_bookstore_agent",
    model=adk_model,
    description="A simple agent that works in a bookstore.",
    instruction="You are a helpful book store assistant. You help the customer by looking up information about books. Use tools, if you need them to fulfill user requests.",
    tools=[read_book_database],
)

In [13]:
# This cell sets up our session aka. conversation.
# If you want to start a new conversation, you need to execute this cell again

# We need to define an app name
simple_bookstore_app_name = "simple_bookstore_app"

# We need to create a runner for the agent
runner = Runner(
    agent=simple_bookstore_agent,
    app_name=simple_bookstore_app_name,
    session_service=session_service
)

# Create a new session id
session_id = create_random_session_id()

# Create the specific session where the conversation will happen
session = await session_service.create_session(
    app_name=simple_bookstore_app_name,
    user_id=USER_ID,
    session_id=session_id
)

In [14]:
# Run the agent. If you need to run this again, make sure you created a new session by running the above cell again

# TODO: Define a prompt that requires the agent to look up book information
user_prompt = "What books by Arthur Conan Doyle do you have? Please provide the title and ISBN."

await run_conversation(session_id=session_id, user_prompt=user_prompt, runner=runner)


<<< User Query: What books by Arthur Conan Doyle do you have? Please provide the title and ISBN.

>>> Tool Call: read_book_database({})
<<< Tool Result: [
    {
        "metadata": {
            "url": "https://www.gutenberg.org/ebooks/30",
            "title": "The Bible, King James Version, Complete",
            "pub_year": 1992,
            "isbn"...

<<< Agent Response: I have the following book by Arthur Conan Doyle:
* Title: A Study in Scarlet, ISBN: 978-3-51593-12345-6


# Exercise 2: Agentic RAG — Bookstore Bot with Sherlock Holmes expertise
In this exercise you should create a new agent, which is an expert for Sherlock Holmes in addition to the book lookup capabilities from exercise 1. The knowledge about Sherlock Holmes can be gathered by using part of our RAG system — the retriever — as a tool. Your task is to create this tool and the respective agent. Afterwards run the agent with the provided user input!

In [15]:
# TODO: Define a sherlock holmes retrieval tool here. Make sure to include a proper docstring/explanation.
def sherlock_holmes_retrieval(query: str) -> list[str]:
    """Retrieves detailed information about Sherlock Holmes books about the query.

    Parameters
      query: str
          The query to be used for the retrieval, query format should match chunks in the knowledgebase
    Returns
      list[str]: The fetched chunks with metadata as strings
    """

    # TODO: Do the retrieval (see helper functions at the top of the notebook)
    chunks = do_retrieval(user_input=query, top_k=10)
    return [
        f"Book: {chunk.metadata.book}, Page: {chunk.metadata.pages}\nContent: {chunk.chunk}"
        for chunk in chunks
    ]

# TODO: Create the advanced bookstore agent. Provide tools, description and instruction
advanced_bookstore_agent = Agent(
    name="advanced_bookstore_agent",
    model=adk_model,
    description="An agent that works in a bookstore and can access detailed information about Sherlock Holmes books.",
    instruction="You are a helpful book store assistant. You help the customer by looking up books and answering questions about Sherlock Holmes books. Use tools, if you need them to fulfill user requests.",
    tools=[sherlock_holmes_retrieval, read_book_database],
)

In [16]:
# This cell sets up our session aka. conversation.
# If you want to start a new conversation, you need to execute this cell again

# We need to define an app name
advanced_bookstore_app_name = "advanced_bookstore_app"

# We need to create a runner for the agent
runner = Runner(
    agent=advanced_bookstore_agent,
    app_name=advanced_bookstore_app_name,
    session_service=session_service
)
# Create a new session id
session_id = create_random_session_id()

# Create the specific session where the conversation will happen
session = await session_service.create_session(
    app_name=advanced_bookstore_app_name,
    user_id=USER_ID,
    session_id=session_id
)

In [25]:
# TODO: This prompt should be responded successfully using the sherlock_holmes_retrieval tool
user_prompt = "In a Sherlock Holmes story Lucy noticed a number on the ceiling when taking breakfast. Which number was written into the ceiling?"

# Run the agent. If you need to run this again, make sure you created a new session by running the above cell again
await run_conversation(session_id=session_id, user_prompt=user_prompt, runner=runner)


<<< User Query: In a Sherlock Holmes story Lucy noticed a number on the ceiling when taking breakfast. Which number was written into the ceiling?

<<< Agent Response: I'm sorry, but I still cannot find any record of a character named Lucy noticing a number on the ceiling during breakfast in a Sherlock Holmes story within my current database.

It's possible this detail is from a specific short story, a lesser-known novel, a particular adaptation of Sherlock Holmes, or I simply do not have that level of detail in my book summaries. If you have the title of the specific story, I might be able to find more information, but with the current details, I am unable to answer your question.


# (Optional) Exercise 3: MCP Integration

In exercises 1 and 2, tools were defined as Python functions *inside* the notebook. In production systems, tools often live in **separate services**. The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) is an open standard that lets agents discover and call tools hosted by external servers.

In this exercise you will:
1. Inspect the MCP server in `bookstore_mcp_server.py` — it exposes `order_book` and `read_book_database` as MCP tools via stdio. This adds the **ordering capability** that was not available in the previous exercises.
2. Create an ADK agent that connects to this MCP server using `McpToolset` instead of defining the tools locally.

**How it works**:  
ADK's `McpToolset` spawns the MCP server as a subprocess and communicates with it over stdin/stdout. The agent discovers the available tools automatically — you don't need to define them in the notebook.

**Hints**:
* Use `McpToolset` with `StdioConnectionParams` and `StdioServerParameters` to connect to the server.
* The server script is at `bookstore_mcp_server.py` in the same directory as this notebook.
* Use `os.path.abspath("bookstore_mcp_server.py")` to get the absolute path.

In [26]:
# TODO: Create an MCP-powered bookstore agent
# The MCP server at bookstore_mcp_server.py provides order_book and read_book_database
# as external tools via the Model Context Protocol.

mcp_server_path = os.path.abspath("bookstore_mcp_server.py")

mcp_bookstore_agent = Agent(
    name="mcp_bookstore_agent",
    model=adk_model,
    description="A bookstore agent that uses MCP tools from an external server.",
    instruction="You are a helpful book store assistant. You help the customer by looking up and ordering books. Use tools, if you need them to fulfill user requests.",
    tools=[
        McpToolset(
            connection_params=StdioConnectionParams(
                server_params=StdioServerParameters(
                    command="uv",
                    args=["run", "python3", mcp_server_path],
                ),
            )
        )
    ],
)

In [19]:
# This cell sets up our session aka. conversation.
# If you want to start a new conversation, you need to execute this cell again

mcp_bookstore_app_name = "mcp_bookstore_app"

runner = Runner(
    agent=mcp_bookstore_agent,
    app_name=mcp_bookstore_app_name,
    session_service=session_service
)

session_id = create_random_session_id()

session = await session_service.create_session(
    app_name=mcp_bookstore_app_name,
    user_id=USER_ID,
    session_id=session_id
)

In [20]:
# Run the MCP-powered agent — now the agent can also ORDER books via the MCP server.
# This ordering capability was not available in exercises 1 and 2.
user_prompt = "I like to order the book 'A Study in Scarlett' by Arthur Conan Doyle. Please do this for me!"

await run_conversation(session_id=session_id, user_prompt=user_prompt, runner=runner)


<<< User Query: I like to order the book 'A Study in Scarlett' by Arthur Conan Doyle. Please do this for me!

>>> Tool Call: read_book_database({})
<<< Tool Result: {'content': [{'type': 'text', 'text': '[\n    {\n        "metadata": {\n            "url": "https://www.gutenberg.org/ebooks/30",\n            "title": "The Bible, King James Version, Complete",\n    ...

>>> Tool Call: order_book({'isbn': '978-3-51593-12345-6'})
<<< Tool Result: {'content': [{'type': 'text', 'text': "Success: You ordered 'A Study in Scarlet'!"}], 'structuredContent': {'result': "Success: You ordered 'A Study in Scarlet'!"}, 'isError': False}

<<< Agent Response: Great news! I've successfully ordered 'A Study in Scarlet' by Arthur Conan Doyle for you.
